# 案例 12 · 日志采集断流 + 告警规则失配，告警风暴

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

压测时把 CPU 告警规则阈值改成 1%、取消持续与静默且未恢复，叠加 Logtail 停止，10 分钟内产生 240 条重复告警，真实错误日志同时断流。

## 练习目标

- 用告警值判断“规则坏了”而非“业务坏了”
- 恢复 Logtail 采集并确认延迟回落
- 按规则维度去重并恢复阈值/For/静默/分组
- 用 `alert_dedup.py` 完成自动化处置

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-12-alert")
lab.reset(start_api=True)
lab.start_logagent()
lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_alert_storm()
print("已注入：告警数 =", len(lab.state["alarms"]))


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**采集组件**

In [ ]:
print(lab.sh("systemctl status logtail"))

**告警规则**

In [ ]:
print(lab.aliyun("cms DescribeMetricRuleList"))

**告警历史**

In [ ]:
print(lab.aliyun("cms DescribeAlarmHistory"))

**SLS 采集状态**

In [ ]:
print(lab.aliyun("sls GetLogStore --Project xhs-prod --LogStore note-api"))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
rule = lab.state["alarm_rules"][0]
print("告警总数:", len(lab.state["alarms"]))
print("规则:", {k: rule[k] for k in ("Threshold", "Period", "For", "Silence")})
print("采集:", lab.state["logstore"])
print("判断：告警值远低于业务水位 + 规则阈值 1% → 规则被改坏，先降噪再查采集")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
# 1) 恢复日志采集
lab.start_logagent()

# 2) 按 10 分钟窗口去重
seen, kept = set(), []
for a in lab.state["alarms"]:
    key = (a["RuleName"], a["Time"][:15])
    if key in seen:
        continue
    seen.add(key)
    kept.append(a)
lab.state["alarms"] = kept

# 3) 规则恢复基线并开启分组
rule = lab.state["alarm_rules"][0]
rule.update({"Threshold": 80, "Period": 60, "For": 180, "Silence": 600, "GroupBy": ["instanceId"]})
lab.state["logstore"].update({"Status": "normal", "IngestLagSec": 2})
lab.state["faults"].pop("alert", None)
lab._save()
time.sleep(2)
print("告警数:", len(kept), "| 采集 lag:", lab.logagent_stats().get("lag_bytes"), "bytes")


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('告警已收敛', bool(len(lab.state['alarms']) <= 5))
lab.check('采集延迟恢复', bool(lab.logagent_stats().get('lag_bytes', 10**9) < 2000))
lab.check('规则阈值恢复', bool(lab.state['alarm_rules'][0]['Threshold'] == 80))

## 步骤 6 · 自动化沉淀

脚本按规则聚合告警，恢复 Logtail，按时间窗口去重，并把阈值/For/静默/分组恢复为基线，最后校验告警数与采集延迟。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.start_logagent()
lab.inject_alert_storm()
out = lab.auto('alert_dedup.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. 告警风暴时如何在不丢真实告警的前提下快速降噪？
2. 监控组件自身需要哪些自监控指标？
3. 如何让压测与生产告警规则互不污染？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')